# E-commerce Checkout A/B Test
## SQL Funnel Analysis by Experiment Group

### Business Question

Does the simplified one-page checkout move more eligible, mature exposed users through the ordered journey below?

```text
checkout_view → payment_attempt → purchase
```

This notebook describes treatment-control differences in the funnel. Statistical inference is evaluated in the primary inference notebook.

### Analysis Definition

| Component | Definition |
|---|---|
| Population | Eligible users with a first valid checkout exposure and a complete 24-hour observation window |
| Grain | One row per mature exposed user |
| Denominator | Mature exposed users in each experiment group |
| Payment step | At least one `payment_attempt` after exposure and within 24 hours |
| Purchase step | At least one `purchase` after a qualifying payment attempt and within 24 hours of exposure |
| Comparison | Control versus treatment using identical eligibility and timing rules |

Raw event rows are not valid funnel denominators because one user can generate multiple events and the raw table includes duplicates, invalid traffic, pre-assignment events, cross-assigned users, and immature exposure windows.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = None

for path in [START_DIR, *START_DIR.parents]:
    if (
        path.name == "week2_checkout_experiment"
        and (path / "data" / "raw").is_dir()
    ):
        PROJECT_ROOT = path
        break

    candidate = path / "week2_checkout_experiment"
    if (candidate / "data" / "raw").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the week2_checkout_experiment directory."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
SQL_PATH = PROJECT_ROOT / "sql" / "02_funnel_analysis.sql"

print("Project root:", PROJECT_ROOT.name)
print("SQL file:", SQL_PATH.relative_to(PROJECT_ROOT))

Project root: week2_checkout_experiment
SQL file: sql/02_funnel_analysis.sql


## 1. Load the Raw Tables

The notebook creates an in-memory SQLite database from the versioned raw CSV files. This keeps the analysis portable and avoids committing a generated `.db` file.

In [2]:
conn = sqlite3.connect(":memory:")

table_names = [
    "users",
    "experiment_assignments",
    "events",
    "orders",
]

raw_table_audit = []

for table_name in table_names:
    table = pd.read_csv(RAW_DATA_DIR / f"{table_name}.csv")
    table.to_sql(
        table_name,
        conn,
        index=False,
        if_exists="replace",
    )
    raw_table_audit.append(
        {"table_name": table_name, "rows": len(table)}
    )

raw_table_audit = pd.DataFrame(raw_table_audit)
print(raw_table_audit.to_string(index=False))

            table_name  rows
                 users 20000
experiment_assignments 20060
                events 36813
                orders  5043


## 2. Execute the Funnel Transformation

The SQL script rebuilds the canonical assignment, event-deduplication, first-exposure, and maturity logic before constructing the ordered user-level funnel. Materialized temporary tables and indexes prevent SQLite from repeatedly recalculating the same validation pipeline.

In [3]:
funnel_sql = SQL_PATH.read_text(encoding="utf-8")
conn.executescript(funnel_sql)

print("Canonical funnel SQL executed successfully.")

Canonical funnel SQL executed successfully.


## 3. Validate Population, Grain, and Ordering

The final user-level table must reconcile to the canonical mature population. It must also preserve a one-user-one-row grain and cumulative step ordering.

In [4]:
population_validation_query = """
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT user_id) AS unique_users,
    SUM(
        CASE WHEN reached_purchase > reached_payment_attempt
             THEN 1 ELSE 0 END
    ) AS invalid_flag_order,
    SUM(
        CASE
            WHEN payment_attempt_timestamp IS NOT NULL
             AND datetime(payment_attempt_timestamp)
                 < datetime(exposure_timestamp)
            THEN 1 ELSE 0
        END
    ) AS pre_exposure_payment_attempts,
    SUM(
        CASE
            WHEN purchase_timestamp IS NOT NULL
             AND datetime(purchase_timestamp)
                 < datetime(payment_attempt_timestamp)
            THEN 1 ELSE 0
        END
    ) AS purchases_before_payment
FROM user_level_funnel;
"""

population_validation = pd.read_sql_query(
    population_validation_query,
    conn,
)

print(population_validation.to_string(index=False))

assert population_validation.loc[0, "rows"] == 15_467
assert population_validation.loc[0, "unique_users"] == 15_467
assert population_validation.loc[0, "invalid_flag_order"] == 0
assert population_validation.loc[0, "pre_exposure_payment_attempts"] == 0
assert population_validation.loc[0, "purchases_before_payment"] == 0

 rows  unique_users  invalid_flag_order  pre_exposure_payment_attempts  purchases_before_payment
15467         15467                   0                              0                         0


In [5]:
group_size_validation = pd.read_sql_query(
    """
    SELECT experiment_group, COUNT(*) AS users
    FROM user_level_funnel
    GROUP BY experiment_group
    ORDER BY experiment_group;
    """,
    conn,
)

print(group_size_validation.to_string(index=False))

expected_group_sizes = {"control": 7_754, "treatment": 7_713}
actual_group_sizes = dict(
    zip(
        group_size_validation["experiment_group"],
        group_size_validation["users"],
    )
)
assert actual_group_sizes == expected_group_sizes

experiment_group  users
         control   7754
       treatment   7713


## 4. Inspect the User-Level Analysis Grain

Each row has one stable experiment group, one exposure timestamp, and binary cumulative funnel flags. Exposure-level device, traffic source, and user type are retained for the segment analysis.

In [6]:
user_level_sample = pd.read_sql_query(
    """
    SELECT
        user_id,
        experiment_group,
        exposure_timestamp,
        payment_attempt_timestamp,
        purchase_timestamp,
        reached_checkout_view,
        reached_payment_attempt,
        reached_purchase
    FROM user_level_funnel
    ORDER BY user_id
    LIMIT 10;
    """,
    conn,
)

print(user_level_sample.to_string(index=False))

user_id experiment_group            exposure_timestamp     payment_attempt_timestamp purchase_timestamp  reached_checkout_view  reached_payment_attempt  reached_purchase
U000001          control 2026-07-10 03:55:57.474342798 2026-07-10 04:17:57.474342798               None                      1                        1                 0
U000002        treatment 2026-07-08 19:53:25.247435122                          None               None                      1                        0                 0
U000003          control 2026-07-12 14:02:33.743649830                          None               None                      1                        0                 0
U000004        treatment 2026-07-09 15:21:40.318484834 2026-07-09 15:25:40.318484834               None                      1                        1                 0
U000005          control 2026-07-14 08:46:27.258509648 2026-07-14 08:59:27.258509648               None                      1                        

## 5. Funnel Results by Experiment Group

`step_to_step_conversion` uses the immediately preceding cumulative step as its denominator. `conversion_from_exposure` always uses mature exposed users in the same experiment group.

In [7]:
funnel_by_step = pd.read_sql_query(
    """
    SELECT *
    FROM funnel_by_step
    ORDER BY experiment_group, step_order;
    """,
    conn,
)

print(funnel_by_step.to_string(index=False))

for _, group_funnel in funnel_by_step.groupby("experiment_group"):
    assert group_funnel.sort_values("step_order")["step_users"].is_monotonic_decreasing

experiment_group  step_order     funnel_step  step_users  previous_step_users  step_to_step_conversion  conversion_from_exposure
         control           1   checkout_view        7754                 7754                   1.0000                    1.0000
         control           2 payment_attempt        5844                 7754                   0.7537                    0.7537
         control           3        purchase        2130                 5844                   0.3645                    0.2747
       treatment           1   checkout_view        7713                 7713                   1.0000                    1.0000
       treatment           2 payment_attempt        6004                 7713                   0.7784                    0.7784
       treatment           3        purchase        2285                 6004                   0.3806                    0.2963


In [8]:
funnel_summary = pd.read_sql_query(
    """
    SELECT *
    FROM funnel_summary
    ORDER BY experiment_group;
    """,
    conn,
)

print(funnel_summary.to_string(index=False))

experiment_group  exposed_users  payment_attempt_users  purchase_users  checkout_to_payment_conversion  payment_to_purchase_conversion  checkout_to_purchase_conversion
         control           7754                   5844            2130                          0.7537                          0.3645                           0.2747
       treatment           7713                   6004            2285                          0.7784                          0.3806                           0.2963


## 6. Descriptive Treatment-Control Comparison

Absolute difference is `treatment rate − control rate`. Relative lift is `treatment rate / control rate − 1`. These are descriptive estimates, not significance tests.

In [9]:
funnel_comparison = pd.read_sql_query(
    """
    SELECT
        metric,
        control_rate,
        treatment_rate,
        absolute_difference,
        relative_lift
    FROM funnel_comparison
    ORDER BY metric_order;
    """,
    conn,
)

print(funnel_comparison.to_string(index=False))

                         metric  control_rate  treatment_rate  absolute_difference  relative_lift
 checkout_to_payment_conversion        0.7537          0.7784               0.0248         0.0328
 payment_to_purchase_conversion        0.3645          0.3806               0.0161         0.0442
checkout_to_purchase_conversion        0.2747          0.2963               0.0216         0.0785


## Interpretation

The treatment group had higher descriptive conversion at both transitions:

- checkout to payment attempt: **75.37% → 77.84%** (**+2.48 percentage points**);
- payment attempt to purchase: **36.45% → 38.06%** (**+1.61 percentage points**); and
- checkout exposure to purchase: **27.47% → 29.63%** (**+2.16 percentage points**, **+7.85% relative lift**).

The pattern is consistent with reduced friction before payment and a modest improvement after payment. This descriptive analysis does not establish statistical significance or justify a launch decision. The primary inference notebook tests the primary user-level purchase outcome, calculate a confidence interval, and distinguish statistical from practical significance.

## Quality Checks

- The final table contains exactly one row per mature exposed user.
- Control and treatment use identical eligibility and 24-hour maturity rules.
- Duplicate assignment and event records are removed before funnel construction.
- Cross-assigned users and internal, test, and bot traffic are excluded.
- Payment occurs after exposure; purchase occurs after payment.
- Funnel user counts are cumulative and monotonically non-increasing.
- No p-value or launch claim is made before the planned primary test.

In [10]:
conn.close()
print("In-memory SQLite connection closed.")

In-memory SQLite connection closed.
